In [29]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np




In [8]:
df = pd.read_csv(r"C:\Grado BData\Bdata2\Reto 7\Reto 7\25-26_R07_NARANJA\Datos\Transformados\df_limpio.csv")

In [9]:
df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Fiador,Impago,Prima,Tipo_Jornada_Laboral_Autónomo,Tipo_Jornada_Laboral_Desempleado,Tipo_Jornada_Laboral_Jornada completa,Tipo_Jornada_Laboral_Tiempo parcial,Estado_Civil_Casado,Estado_Civil_Divorciado,Estado_Civil_Soltero
0,S97R7X,18,16000,5000,397,19,1,8.06,48,0.10,...,0,0,50.12,1,0,0,0,0,0,1
1,RLGTBY,50,62116,37278,486,217,3,21.96,12,0.55,...,1,1,800.00,0,0,0,1,1,0,0
2,SKE2P9,37,37602,44532,765,150,3,11.20,60,0.23,...,1,0,356.30,0,0,0,1,1,0,0
3,E2FB1D,56,67410,23752,643,369,1,21.24,24,0.18,...,0,0,198.72,1,0,0,0,1,0,0
4,TKSCGH,35,35930,28440,645,136,3,16.95,12,0.55,...,1,0,484.20,0,0,1,0,0,0,1


In [10]:
paleta_colores = {
    'rojo_220': '#A50050',   
    'verde_376': '#84BD00',   
    'morado_262': '#51284F',  
    'gris_9043': '#EAE7E0'   
}

In [11]:
#GRÁFICO 1: Distribución de la variable objetivo
fig1 = px.histogram(
    df, 
    x='Impago',
    title='Distribución de préstamos con impago',
    labels={'Impago': 'Estado del préstamo (0=Pagado, 1=Impago)', 'count': 'Número de préstamos'},
    color='Impago',
    color_discrete_map={0: '#84BD00', 1: '#A50050'},
    text_auto=True
)

#Calcular y añadir porcentajes
total = len(df)
counts = df['Impago'].value_counts()
for i, val in enumerate([0, 1]):
    fig1.add_annotation(
        x=val, 
        y=counts[val],
        text=f"{counts[val]/total*100:.1f}%",
        showarrow=False,
        yshift=15,
        font=dict(size=14)
    )

fig1.update_layout(
    xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=['Pagado', 'Impago']),
    showlegend=False,
    width=700,
    height=500
)
fig1.show()

In [12]:
#GRÁFICO 2: Relación entre ingresos y monto del préstamo

#Usamos sample para no saturar (3000 puntos aleatorios)
sample_df = df.sample(n=3000, random_state=42)

fig2 = px.scatter(
    sample_df,
    x='Ingresos',
    y='Monto_Inicial',
    color='Impago',
    color_discrete_map={0: '#84BD00', 1: '#A50050'},
    opacity=0.6,
    title='Relación entre Ingresos y Monto del Préstamo',
    labels={
        'Ingresos': 'Ingresos anuales (€)',
        'Monto_Inicial': 'Monto inicial del préstamo (€)',
        'Impago': 'Estado'
    }
)

fig2.update_layout(
    width=900,
    height=600,
    legend_title_text='Estado'
)

#Personalizar la leyenda
fig2.update_layout(legend=dict(
    title="Estado",
    itemsizing='constant',
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=0.99
))

fig2.show()

In [13]:
#GRÁFICO 3: Distribución del scoring crediticio por impago
fig3 = px.box(
    df,
    x='Impago',
    y='Scoring_Crediticio',
    color='Impago',
    color_discrete_map={0: '#84BD00', 1: '#A50050'},
    title='Distribución del scoring crediticio por estado de impago',
    labels={
        'Impago': 'Estado del préstamo',
        'Scoring_Crediticio': 'Puntuación crediticia'
    },
    points=False
)

fig3.update_layout(
    xaxis=dict(tickmode='array', tickvals=[0, 1], ticktext=['Pagado', 'Impago']),
    showlegend=False,
    width=800,
    height=600
)
fig3.show()

In [14]:
#GRÁFICO 4: Tasa de impago por nivel de estudios
# Calcular tasa de impago por estudios
impago_estudios = df.groupby('Estudios')['Impago'].agg(['mean', 'count']).reset_index()
impago_estudios.columns = ['Estudios', 'Tasa_Impago', 'Total']

#Asegurar que Estudios sea tratado como categoría (string)
impago_estudios['Estudios'] = impago_estudios['Estudios'].astype(str)

#Ordenar por nivel de estudios (0,1,2,3)
impago_estudios = impago_estudios.sort_values('Estudios', ascending=True)

fig4 = px.bar(
    impago_estudios,
    x='Estudios',
    y='Tasa_Impago',
    title='Tasa de impago por nivel de estudios',
    labels={
        'Tasa_Impago': 'Tasa de impago',
        'Estudios': 'Nivel de estudios'
    },
    text=impago_estudios['Tasa_Impago'].apply(lambda x: f'{x:.1%}'),
    category_orders={'Estudios': ['0', '1', '2', '3']}  # Forzar orden 0,1,2,3
)

# Asignar un color distinto de la paleta a cada barra
colores = ['#A50050', '#84BD00', '#51284F', '#EAE7E0']  # rojo, verde, morado, gris
fig4.update_traces(
    textposition='outside',
    marker_color=colores
)

fig4.update_layout(
    width=900,
    height=500,
    yaxis_tickformat='.0%'
)
fig4.show()

In [15]:
#GRÁFICO 5: Relación entre duración y tasa de impago
#Crear categorías de duración (tramos de 12 meses)
df['Duracion_Tramo'] = pd.cut(df['Duracion'], bins=[0, 12, 24, 36, 48, 60, 72], 
                               labels=['0-12', '13-24', '25-36', '37-48', '49-60', '61-72'])

#Calcular tasa de impago por tramo
tasa_duracion = df.groupby('Duracion_Tramo', observed=True)['Impago'].mean().reset_index()

fig5 = px.line(
    tasa_duracion,
    x='Duracion_Tramo',
    y='Impago',
    markers=True,
    title='Tasa de impago por duración del préstamo',
    labels={
        'Duracion_Tramo': 'Duración del préstamo (meses)',
        'Impago': 'Tasa de impago'
    }
)

# Cambiamos el color de la línea y los marcadores a rojo_220
fig5.update_traces(line=dict(color='#A50050', width=3), marker=dict(color='#A50050', size=10))

fig5.update_layout(
    width=900,
    height=500,
    yaxis_tickformat='.1%'
)
fig5.show()

In [16]:
#GRÁFICO 6: Matriz de correlación
#Seleccionar variables numéricas relevantes
num_vars = ['Edad', 'Ingresos', 'Monto_Inicial', 'Scoring_Crediticio', 
            'Meses_Empleo', 'Num_Creditos', 'Ratio_Interes', 
            'Duracion', 'Ratio_Deuda_Ingresos', 'Prima', 'Impago']

#Calcular correlaciones
corr_matrix = df[num_vars].corr()

fig7 = px.imshow(
    corr_matrix,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    title='Matriz de Correlación - Variables Numéricas',
    labels=dict(color='Correlación'),
    zmin=-1, zmax=1
)

fig7.update_layout(
    width=900,
    height=800
)
fig7.show()

In [17]:
#GRÁFICO 7: Distribución del ratio de interés por impago
fig8 = px.histogram(
    df,
    x='Ratio_Interes',
    color='Impago',
    color_discrete_map={0: '#84BD00', 1: '#A50050'},
    opacity=0.7,
    title='Distribución del ratio de interés por estado de impago',
    labels={
        'Ratio_Interes': 'Ratio de interés (%)',
        'count': 'Frecuencia'
    },
    barmode='overlay',
    nbins=50
)

fig8.update_layout(
    width=900,
    height=600,
    legend_title_text='Impago'
)
fig8.show()

In [18]:
#GRÁFICO 8: Ingresos vs Prima del seguro
#Crear categorías de ingresos (quintiles)
df['Categoria_Ingresos'] = pd.qcut(df['Ingresos'], q=5, labels=['Muy bajos', 'Bajos', 'Medios', 'Altos', 'Muy altos'])

#Calcular prima media por categoría
prima_ingresos = df.groupby('Categoria_Ingresos', observed=True)['Prima'].mean().reset_index()

fig9 = px.bar(
    prima_ingresos,
    x='Categoria_Ingresos',
    y='Prima',
    title='Prima media del seguro por nivel de ingresos',
    labels={
        'Categoria_Ingresos': 'Categoría de ingresos',
        'Prima': 'Prima media (€)'
    },
    text=prima_ingresos['Prima'].apply(lambda x: f'€{x:.1f}')
)

# Asignar un color distinto de la paleta a cada barra
colores = ['#A50050', '#84BD00', '#51284F', '#EAE7E0', '#A50050']  # rojo, verde, morado, gris, rojo
fig9.update_traces(textposition='outside', marker_color=colores)

fig9.update_layout(
    width=900,
    height=500,
    coloraxis_showscale=False
)
fig9.show()

In [19]:
#GRÁFICO 9: Propósito del préstamo y tasa de impago
#Calcular tasa de impago por propósito
impago_fiador = df.groupby('Fiador')['Impago'].mean().reset_index()
impago_fiador.columns = ['Fiador', 'Tasa_Impago']
impago_fiador['Fiador'] = impago_fiador['Fiador'].map({0: 'Sin fiador', 1: 'Con fiador'})

fig_fiador = px.bar(
    impago_fiador,
    x='Fiador',
    y='Tasa_Impago',
    title='Tasa de impago por presencia de fiador',
    labels={
        'Fiador': 'Fiador',
        'Tasa_Impago': 'Tasa de impago'
    },
    text=impago_fiador['Tasa_Impago'].apply(lambda x: f'{x:.1%}')
)

# Asignar un color distinto de la paleta a cada barra
colores = ['#A50050', '#84BD00']  # rojo = Sin fiador, verde = Con fiador
fig_fiador.update_traces(textposition='outside', marker_color=colores)

fig_fiador.update_layout(
    width=700,
    height=500,
    coloraxis_showscale=False,
    yaxis_tickformat='.0%'
)
fig_fiador.show()

In [20]:
#GRÁFICO 10: Distribución de la prima del seguro
fig11 = px.histogram(
    df,
    x='Prima',
    nbins=50,
    title='Distribución de la prima del seguro',
    labels={'Prima': 'Prima (€)', 'count': 'Frecuencia'},
    color_discrete_sequence=['#A50050']  # rojo_220
)

fig11.update_layout(width=900, height=500)
fig11.show()

In [21]:
#GRÁFICO 11: Tasa de impago por tramos de ingresos

#Crear tramos de ingresos
df['tramo_ingresos'] = pd.cut(df['Ingresos'], bins=5, labels=['Muy bajos', 'Bajos', 'Medios', 'Altos', 'Muy altos'])

#Calcular tasa de impago por tramo
tasa_por_ingreso = df.groupby('tramo_ingresos', observed=True)['Impago'].mean().reset_index()

fig12_ingresos = px.bar(
    tasa_por_ingreso,
    x='tramo_ingresos',
    y='Impago',
    title='Tasa de impago por nivel de ingresos',
    labels={
        'tramo_ingresos': 'Nivel de ingresos',
        'Impago': 'Tasa de impago'
    },
    text=tasa_por_ingreso['Impago'].apply(lambda x: f'{x:.1%}')
)

# Asignar un color distinto de la paleta a cada barra
colores = ['#A50050', '#84BD00', '#51284F', '#EAE7E0', '#A50050']
fig12_ingresos.update_traces(textposition='outside', marker_color=colores)

fig12_ingresos.update_layout(
    width=700,
    height=500,
    coloraxis_showscale=False,
    yaxis_tickformat='.0%'
)
fig12_ingresos.show()

In [22]:
#GRÁFICO 12: Relación entre ingresos, monto y scoring crediticio

#Muestra para no saturar
sample_df = df.sample(n=1000, random_state=42)

#Crear columna con etiquetas para la leyenda
sample_df['estado_texto'] = sample_df['Impago'].map({0: 'Pagado', 1: 'Impago'})

fig_burbujas = px.scatter_3d(
    sample_df,
    x='Ingresos',
    y='Monto_Inicial',
    z='Scoring_Crediticio',
    color='estado_texto',
    color_discrete_map={'Pagado': '#84BD00', 'Impago': '#A50050'},  # colores de la paleta
    size='Prima',
    opacity=0.7,
    title='Relación entre ingresos, monto y scoring crediticio',
    labels={
        'Ingresos': 'Ingresos anuales (€)',
        'Monto_Inicial': 'Monto del préstamo (€)',
        'Scoring_Crediticio': 'Scoring crediticio',
        'Prima': 'Prima del seguro',
        'estado_texto': 'Estado'
    }
)

#Eliminar barra de color
fig_burbujas.update_coloraxes(showscale=False)

#Personalizar leyenda
fig_burbujas.update_layout(
    width=900,
    height=700,
    legend=dict(
        title="Estado del préstamo",
        itemsizing='constant',
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

fig_burbujas.update_traces(marker=dict(size=4))
fig_burbujas.show()

In [ ]:
#GRÁFICO 13: Piramide poblacional 
#Crear grupos de edad (bins de 5 años)
bins_edad = list(range(15, 81, 5))
labels_edad = [f"{i}-{i+4}" for i in bins_edad[:-1]]
df['grupo_edad'] = pd.cut(df['Edad'], bins=bins_edad, labels=labels_edad, right=False)

#Calcular porcentajes por grupo de edad y estado
edad_pagado = df[df['Impago'] == 0].groupby('grupo_edad', observed=True).size()
edad_impago = df[df['Impago'] == 1].groupby('grupo_edad', observed=True).size()

#Normalizar para que sumen 100% dentro de cada estado
total_pagado = edad_pagado.sum()
total_impago = edad_impago.sum()

edad_pagado_pct = (edad_pagado / total_pagado * 100).round(1)
edad_impago_pct = (edad_impago / total_impago * 100).round(1)

#Crear figura con subplots
fig = make_subplots(rows=1, cols=2, 
                    subplot_titles=('Pagado', 'Impago'),
                    shared_yaxes=True,
                    horizontal_spacing=0.05)

#Añadir barras para pagado (izquierda, negativas)
fig.add_trace(
    go.Bar(
        x=-edad_pagado_pct.values,
        y=labels_edad,
        orientation='h',
        name='Pagado',
        marker=dict(color=paleta_colores['verde_376']),
        text=edad_pagado_pct.values,
        texttemplate='%{text:.1f}%',
        textposition='inside',
        hovertemplate='Edad: %{y}<br>Pagados: %{text}%<extra></extra>'
    ),
    row=1, col=1
)

#Añadir barras para impago (derecha, positivas)
fig.add_trace(
    go.Bar(
        x=edad_impago_pct.values,
        y=labels_edad,
        orientation='h',
        name='Impago',
        marker=dict(color=paleta_colores['rojo_220']),
        text=edad_impago_pct.values,
        texttemplate='%{text:.1f}%',
        textposition='inside',
        hovertemplate='Edad: %{y}<br>Impagos: %{text}%<extra></extra>'
    ),
    row=1, col=2
)

#Personalizar layout
fig.update_layout(
    title='<b>Pirámide poblacional: distribución de edad por estado del préstamo</b><br>' +
          '<sup>Porcentaje sobre el total de cada grupo (pagado vs impago)</sup>',
    barmode='overlay',
    width=1000,
    height=600,
    plot_bgcolor=paleta_colores['gris_9043'],
    paper_bgcolor='white',
    showlegend=False,
    hovermode='y unified'
)

#Ajustar ejes
fig.update_xaxes(
    title_text='Porcentaje (%)',
    range=[-25, 0],
    tickvals=[-20, -15, -10, -5, 0],
    ticktext=['20%', '15%', '10%', '5%', '0%'],
    row=1, col=1
)

fig.update_xaxes(
    title_text='Porcentaje (%)',
    range=[0, 25],
    tickvals=[0, 5, 10, 15, 20],
    ticktext=['0%', '5%', '10%', '15%', '20%'],
    row=1, col=2
)

fig.update_yaxes(
    title_text='Grupo de edad',
    row=1, col=1
)

#Añadir línea central
fig.add_shape(
    type='line',
    x0=0, y0=-0.5,
    x1=0, y1=len(labels_edad)-0.5,
    line=dict(color=paleta_colores['morado_262'], width=1, dash='dash'),
    row=1, col=1
)

fig.add_shape(
    type='line',
    x0=0, y0=-0.5,
    x1=0, y1=len(labels_edad)-0.5,
    line=dict(color=paleta_colores['morado_262'], width=1, dash='dash'),
    row=1, col=2
)

fig.show()
